# 06 · R0 — Kiểm chứng pipeline: tái lập TextVQA của Qwen2.5-VL-3B (W7)

**Vấn đề:** B1 (baseline zero-shot trên ViVQA/ViTextVQA) chưa thể đối chiếu với paper, vì chưa paper nào công bố Qwen2.5-VL-3B trên hai bộ này. Gần nhất là Bảng 7 của paper ViTextVQA (QwenVL-7b, 100 câu, không công bố checkpoint và prompt), nên không tái lập được. Vậy phải trả lời câu hỏi khác: **code của nhóm có tái lập được một con số chính thức của chính model này không?**

**Mốc so sánh:** Qwen2.5-VL Technical Report (Bai et al., 2025, arXiv:2502.13923), **Bảng 5: Qwen2.5-VL-3B, TextVQA val = 79,3**.

| Biến thể | Cấu hình | Mục đích |
|---|---|---|
| **R0** (`paper`) | bf16, **không** lượng tử hoá, độ phân giải mặc định của processor | So với 79,3 → kiểm chứng pipeline (nạp model, xử lý ảnh, sinh, chấm) |
| **R0b** (`b1cfg`) | Đúng cấu hình B1: 4-bit NF4, bf16, 512 token ảnh | Đo **cái giá** của cấu hình tiết kiệm (bằng chứng cho chữ "Efficient" trong đề tài) |

Cả hai: toàn bộ **5.000 câu TextVQA val** (HF `lmms-lab/textvqa`, ghi revision), greedy, `max_new_tokens`=32, prompt theo quy ước lmms-eval `"<câu hỏi>\nAnswer the question using a single word or phrase."` (report không công bố prompt). Metric: **VQA accuracy chính thức** (chuẩn hoá EvalAI, trung bình leave-one-out trên 10 đáp án).

**Luật (chốt trước khi chạy):** R0 **tái lập được** nếu |VQA-Acc − 79,3| ≤ **2,0 điểm** (sai số chuẩn của 5.000 câu ≈ 0,6 điểm; nới thêm cho khác biệt prompt/độ phân giải mà report không công bố). Nếu lệch: **không** sửa cấu hình cho tới khi khớp; ghi lại, rồi điều tra nguyên nhân (prompt, độ phân giải, cách chấm) như một phát hiện.

**Thời gian ước tính:** tải dữ liệu vài phút; R0 khoảng 30–40 phút; R0b khoảng 35–45 phút → tổng **khoảng 1–1,5 giờ, khoảng 2–2,5 CU**. Chạy **sau khi B2 xong** (một GPU một việc).

**Nếu Colab rớt:** chạy lại từ phần 1 rồi Run all; các lô 500 câu đã xong tự bỏ qua.

## 1. Thiết lập môi trường

Giống notebook 02/04 (cùng pin `transformers==5.17.0`). Dừng hẳn nếu GPU không phải L4.

*Kỳ vọng: commit mới nhất, `GPU: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml pyarrow

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (lmms-lab/textvqa là dataset công khai, vẫn tải được)')

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU'
print('GPU:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ R0 phải chạy trên L4 (cùng GPU với B1/B2). Đổi runtime sang L4 rồi chạy lại cell này.')
print('✅ Thiết lập xong')

## 2. Tải TextVQA val + selftest

Tải split validation (parquet) từ HuggingFace, ghi ảnh **nguyên byte gốc** ra `/content/textvqa_val/`, ghi bản chuẩn hoá `val.json` + `prep_report.json` (có revision) lên Drive.

*Kỳ vọng: `5000 câu`, khoảng 3.100+ ảnh, một revision dài 40 ký tự, `SELFTEST OK`.*

In [ ]:
import json
from src.eval import run_textvqa_r0 as r0

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
EXP_ROOT   = f'{DRIVE_ROOT}/experiments'
RUN_R0B    = True    # False: chỉ chạy R0 (tiết kiệm ~40 phút)
RUN_R0     = 'W07_R0_textvqa_val_qwen25vl3b_paper_bf16_L4'
RUN_R0B_N  = 'W07_R0b_textvqa_val_qwen25vl3b_b1cfg_L4'

data = r0.prepare_textvqa(DRIVE_ROOT)
print('câu:', len(data['records']), '| ảnh:', len({x['image_id'] for x in data['records']}),
      '| revision:', data['revision'])
print('ví dụ:', json.dumps(data['records'][0], ensure_ascii=False)[:200])
!python -m src.eval.run_textvqa_r0 --selftest 2>&1 | tail -n 1

## 3. R0 — cấu hình gần paper nhất (≈30–40 phút)

*Kỳ vọng: 10 lô, mỗi lô in `s/câu`; cuối cùng `== KẾT QUẢ: VQA-Acc = … | TÁI LẬP ĐƯỢC …` hoặc `LỆCH …`.*

In [ ]:
if os.path.exists(f'{EXP_ROOT}/{RUN_R0}/metrics.json'):
    print('⏭ R0 đã xong -> bỏ qua')
else:
    !python -u -m src.eval.run_textvqa_r0 --variant paper --run_name {RUN_R0}

## 4. R0b — cấu hình B1 (4-bit, 512 token) trên đúng 5.000 câu đó (≈35–45 phút)

*Kỳ vọng: 10 lô, cuối cùng `== KẾT QUẢ: VQA-Acc = … | không áp luật (biến thể đo chi phí cấu hình B1)`.*

In [ ]:
if not RUN_R0B:
    print('RUN_R0B = False -> bỏ qua')
elif os.path.exists(f'{EXP_ROOT}/{RUN_R0B_N}/metrics.json'):
    print('⏭ R0b đã xong -> bỏ qua')
else:
    !python -u -m src.eval.run_textvqa_r0 --variant b1cfg --run_name {RUN_R0B_N}

## 5. Kết quả

Bảng so với report, kết luận theo luật, và chênh lệch **ghép cặp** R0b − R0 (cùng 5.000 câu) kèm CI95 bootstrap.

*Kỳ vọng: một dòng `KẾT LUẬN R0`, một dòng `Cái giá của cấu hình B1`.*

In [ ]:
import numpy as np
import pandas as pd

def load(run):
    d = f'{EXP_ROOT}/{run}'
    return json.load(open(f'{d}/metrics.json', encoding='utf-8')), json.load(open(f'{d}/predictions.json', encoding='utf-8'))

runs = {'R0 (paper: bf16, không lượng tử, độ phân giải mặc định)': RUN_R0}
if RUN_R0B and os.path.exists(f'{EXP_ROOT}/{RUN_R0B_N}/metrics.json'):
    runs['R0b (cấu hình B1: 4-bit NF4, 512 token)'] = RUN_R0B_N
rows, P = [], {}
for lab, run in runs.items():
    m, p = load(run); P[lab] = p
    rows.append({'run': lab, 'n': m['meta']['num_samples'], 'VQA-Acc (%)': 100 * m['metrics']['vqa_accuracy'],
                 'report (%)': r0.REPORTED['value'], 'chênh': 100 * m['metrics']['vqa_accuracy'] - r0.REPORTED['value'],
                 's/câu': m['meta']['sec_per_sample'], 'VRAM đỉnh (GB)': m['meta']['peak_vram_gb_this_session'],
                 'commit': m['meta']['git_commit']})
display(pd.DataFrame(rows).round(2))
m0, _ = load(RUN_R0)
print('➜ KẾT LUẬN R0:', m0['verdict'])
print('  nguồn:', r0.REPORTED['source'])

if len(P) == 2:
    a, b = list(P.values())
    da = {x['question_id']: x['vqa_acc'] for x in a}
    diff = np.array([x['vqa_acc'] - da[x['question_id']] for x in b]) * 100
    rng = np.random.default_rng(42)
    boots = [diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(10000)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    COST = {'delta_points': float(diff.mean()), 'ci95': [float(lo), float(hi)]}
    print(f"➜ Cái giá của cấu hình B1: {diff.mean():+.2f} điểm VQA-Acc (CI95 [{lo:+.2f}; {hi:+.2f}]) so với R0")
else:
    COST = None

## 6. Soi các câu R0 trả lời sai

10 câu R0 được 0 điểm (seed 0), để xem lỗi là do model hay do cách chấm/prompt.

*Kỳ vọng: phần lớn là lỗi đọc chữ thật hoặc đáp án chuẩn khác cách viết; nếu thấy câu trả lời dài dòng hàng loạt → prompt có vấn đề, báo Claude.*

In [ ]:
import random
a = P[list(P)[0]]
wrong = [x for x in a if x['vqa_acc'] == 0]
print(f'R0 sai hoàn toàn {len(wrong)}/{len(a)} câu')
for x in random.Random(0).sample(wrong, min(10, len(wrong))):
    print(f"Q: {x['question']}\n   PRED: {x['prediction']!r}\n   GOLD: {sorted(set(x['answers']))[:5]}")

## 7. Lưu tóm tắt

Ghi `experiments/W07_R0_summary_L4/r0_summary.json`. Cursor chép file này + `metrics.json`, `predictions.json`, `run_signature.json` của R0/R0b + `data/textvqa_val/prep_report.json` vào repo (không chép `chunks/`, không chép ảnh).

*Kỳ vọng: `Ghi -> …/r0_summary.json`. Sau đó **Disconnect and delete runtime**.*

In [ ]:
OUT_DIR = f'{EXP_ROOT}/W07_R0_summary_L4'
os.makedirs(OUT_DIR, exist_ok=True)
summary = {'what': 'R0: kiểm chứng pipeline bằng tái lập TextVQA val của Qwen2.5-VL-3B',
           'reported': r0.REPORTED, 'tolerance_points': r0.TOLERANCE, 'verdict_R0': m0['verdict'],
           'runs': runs, 'table': rows, 'b1cfg_cost_vs_R0': COST, 'revision': data['revision'], 'gpu': GPU}
with open(f'{OUT_DIR}/r0_summary.json', 'w', encoding='utf-8') as fp:
    json.dump(summary, fp, ensure_ascii=False, indent=2, default=str)
print('Ghi ->', f'{OUT_DIR}/r0_summary.json')
print('➡ Xong: Runtime -> Disconnect and delete runtime, rồi kiểm tra Manage sessions.')